# Module 5: Giving the agent knowledge it can search

> **Prototype notebook.** It walks this module's steps with the evidence kept on the page and links
> straight to the right place in MLflow and the OpenShift console. The wording is a working draft;
> the module's published text is the lab author's.

In [ ]:
import labnb as lab
%cd ../agent

## Start from module 5

This puts the agent in the state module 5 starts from. Any edits you made are copied to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
!python reset.py 5

## 1. An answer the model can't know

Fault F-21 is documented only in Kestrel's service bulletins, which no model has seen.

In [ ]:
!python agent.py --q m5
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))
lab.trace_link()

A specific, plausible answer. Module 4's contract doesn't catch it, because no tool failed. In the
trace, look for a retrieval step: there isn't one, because the agent has no way to search yet.

## 2. The knowledge it could use

The bulletins are already loaded into a PostgreSQL database with pgvector in your project. Query it
directly, outside the agent:

In [ ]:
!python retrieval.py "fault F-21"

Each passage carries its `source`, the bulletin it came from. That's what makes an answer
checkable.

## 3. Register the search tool

In [ ]:
!cp solutions/module-05/tools.py solutions/module-05/agent.py .
!git diff --stat -- tools.py agent.py

`search_bulletins` is declared like the other tools, and returns passages with their sources using
module 4's contract.

```mermaid
flowchart LR
  Q["The F-21 question"] --> M["Model"]
  M -- "search_bulletins: fault F-21" --> T["Retrieval tool"]
  T --> DB[("pgvector: Kestrel service bulletins")]
  DB --> T
  T -- "passages with their source ids, in module 4's contract" --> M
  M --> ANS["An answer that matches a passage"]
```

## 4. Ask again

In [ ]:
!python agent.py --q m5
lab.tool_results()
lab.trace_link()

**This is the module's success signal:** `search_bulletins` was called, passages came back from
SB-2024-102, and the answer follows the passage ("coolant loop pressure low") rather than the
model's guess.

## 5. Ask something the bulletins don't cover

There is no Kestrel K-900.

In [ ]:
!python agent.py --q m5-miss
lab.tool_results()
lab.evidence(columns=("question", "tools run", "answer"), last=3)

The search returns `no_results`, and the agent says so instead of inventing. Compare it with the
first run of this module, which invented.

Grounding makes an answer checkable; it doesn't make it correct. Nothing yet checks that the
retrieved passage actually answers the question. Module 7 comes back to that.